# Exercises week 37

**FYS-STK3155/4155, fall 2026 — deadline Friday September 11 at midnight.**

## Implementing gradient descent for ordinary least squares and Ridge regression

The aim this week is that you write, and *check*, your own gradient descent
code for OLS and Ridge regression, and that you understand what the learning
rate can and cannot be. Everything here is reused directly in part e) of
Project 1, and exercise 6 opens part f). The code developed in the Tuesday
session (`week37tuesday.ipynb`, Case 1) is meant to be reused.

After completing these exercises you will have

* your own code for the simplest gradient descent approach applied to
  ordinary least squares (OLS) and Ridge regression, with a stopping criterion;
* compared the analytical expressions for the OLS and Ridge parameters with
  the gradient descent results;
* explored the role of the learning rate $\eta$ in gradient descent and of the
  hyperparameter $\lambda$ in Ridge regression, and related the largest usable
  learning rate to the largest eigenvalue of the Hessian (Section 4.5 of the
  lecture notes);
* checked your analytical gradient against automatic differentiation
  (Section 4.14) and added momentum (Section 4.6);
* scaled the data properly.

Reading: Chapter 4, Sections 4.1–4.6 and 4.14 of the book; Goodfellow et al.,
Chapter 4 and Sections 8.1–8.3.

## A simple one-dimensional second-order polynomial

We start with a very simple function,

$$
f(x) = 2 - x + 5x^2,
$$

defined for $x \in [-2, 2]$. You can add noise if you wish (the Tuesday
session used Gaussian noise with standard deviation $0.5$). We fit this
function with a polynomial ansatz. The easiest thing is to set up a
second-order polynomial and see if you can recover the above function; feel
free to play around with higher-order polynomials — the Tuesday session shows
that a degree-5 fit of the same data is a much harder problem for gradient
descent, and exercise 4 asks you why.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2026)
n = 100
x = rng.uniform(-2.0, 2.0, n)
noise = 0.5
y = 2.0 - x + 5.0 * x**2 + noise * rng.standard_normal(n)   # drop the noise term if you prefer


## Exercise 1: scale your data

Before fitting a regression model it is good practice to standardise the
features. This ensures all features are on a comparable scale, which is
especially important when using regularisation — and, as Section 4.5 of the
lecture notes shows, it is also what makes gradient descent converge in a
reasonable number of iterations. Here we standardise, scaling each feature to
have mean $0$ and standard deviation $1$.

### 1a)

Set up the design matrix $\boldsymbol{X}$ with the columns $x$ and $x^2$ (no
column of ones). Compute the mean and standard deviation of each column,
subtract the mean and divide by the standard deviation.

We also centre the target $\boldsymbol{y}$ to mean $0$. Centring
$\boldsymbol{y}$ and each feature means the model does not need a separate
intercept term: the data are shifted so that the intercept is effectively
$0$. (In practice one can include an intercept in the model and leave it
unpenalised, see Section 3.13 of the lecture notes; here we simplify by
centring.)

In [214]:
X = np.column_stack([x, x**2])

# Standardize features (zero mean, unit variance for each feature)
X_mean = X.mean(axis=0)
X_std = X.std(axis=0)
X_std[X_std == 0] = 1  # safeguard to avoid division by zero for constant features
X_norm = (X - X_mean) / X_std

# Center the target to zero mean (optional, to simplify intercept handling)
y_mean = y.mean()
y_centered = y - y_mean

# 1b Retrieving the original scale
X_o = (X_norm*X_std + X_mean)

# Coefficients
print(X.shape)
print(X[0,:])
print(X_norm[0,:])
print(X_o[0,:])
print("------")
# Output values
print(y[0])
print(y_centered[0])
print(y_centered[0]+y_mean)

# Etter fitting
# beta_original = theta_fitted / X_std 

(100, 2)
[-1.28426075  1.64932566]
[-1.30860099  0.48289472]
[-1.28426075  1.64932566]
------
10.890107416458507
3.488714340642768
10.890107416458507


Fill in the necessary details. Do we need to centre the $y$-values? What
would go wrong, for OLS and for Ridge, if we did not? 

After this preprocessing each column of $\boldsymbol{X}_{\mathrm{norm}}$ has
mean zero and standard deviation $1$ and $\boldsymbol{y}_{\mathrm{centered}}$
has mean $0$. This makes the optimisation landscape nicer and ensures that the
penalty $\lambda\sum_j\theta_j^2$ in Ridge regression treats each coefficient
fairly, since the features are on the same scale.

### 1b)

The true function has coefficients $(-1, 5)$ on $(x, x^2)$. Which
coefficients do you expect to find on the *standardised* columns, and how do
you convert the fitted parameters back to the original scale?

## Exercise 2: calculate the gradients and the Hessian

Find the gradients of the OLS and Ridge cost functions,

$$
C_{\mathrm{OLS}}(\boldsymbol{\theta}) = \frac{1}{n}\|\boldsymbol{X}\boldsymbol{\theta} - \boldsymbol{y}\|_2^2,
\qquad
C_{\mathrm{Ridge}}(\boldsymbol{\theta}) = \frac{1}{n}\|\boldsymbol{X}\boldsymbol{\theta} - \boldsymbol{y}\|_2^2 + \lambda\boldsymbol{\theta}^T\boldsymbol{\theta},
$$

with respect to $\boldsymbol{\theta}$ (Eqs. (4.13) and (4.17) of the lecture
notes). Find also the Hessian matrices, and show that the OLS Hessian is
positive semi-definite and the Ridge Hessian positive definite for
$\lambda > 0$ — that is, that both cost functions are convex (Section 4.1).
Why does this matter for gradient descent?

Setting the Ridge gradient to zero, show that the minimiser is
$\hat{\boldsymbol{\theta}} = (\boldsymbol{X}^T\boldsymbol{X} + n\lambda\boldsymbol{I})^{-1}\boldsymbol{X}^T\boldsymbol{y}$.
Where does the factor $n$ come from, and what is the corresponding `alpha`
of `scikit-learn`'s `Ridge`? (Section 3.10 and Eq. (3.95).)

## Exercise 3: the analytical formulae for OLS and Ridge regression

In [215]:
# Set regularization parameter, either a single value or a vector of values
# Note that lambda is a python keyword: the lambda keyword creates small anonymous functions.
lam = 0.5

# Analytical forms: theta_Ridge = (X^T X + n*lambda*I)^{-1} X^T y and theta_OLS = (X^T X)^{-1} X^T y
n_features = X_norm.shape[1]
I = np.eye(n_features)
def theta_closed_OLS(X, y):
    return np.linalg.solve(X.T @ X, X.T @ y)
def theta_closed_Ridge(X, y, lmb):
    return np.linalg.solve(X.T @ X + n * lmb * I, X.T @ y)

theta_closed_formOLS =theta_closed_OLS(X_norm, y_centered)
theta_closed_formRidge = theta_closed_Ridge(X_norm, y_centered, lam)

print("Closed-form OLS coefficients:", theta_closed_formOLS)
print("Closed-form Ridge coefficients:", theta_closed_formRidge)

from sklearn.linear_model import Ridge

theta_R_sk = Ridge(alpha=n*lam, fit_intercept=False).fit(X,y).coef_
print("Skicit Ridge coefficients:",theta_R_sk)

Closed-form OLS coefficients: [-1.0773505  5.854734 ]
Closed-form Ridge coefficients: [-0.5608836   3.86236727]
Skicit Ridge coefficients: [-0.59518638  4.88131791]


In [216]:
# 3c , different values of hyperparameter lambda, cross validation
from sklearn.model_selection import train_test_split, KFold, cross_val_score
k = 10
kFold = KFold(n_splits = k, shuffle=True, random_state=2026)
lambdas = np.logspace(-3, 5, n)

def cross_val(x, y):
    score_KFold = np.zeros((n,k))
    for i, lmb in enumerate(lambdas):
        for j, (train_inds, test_inds) in enumerate(kFold.split(x)):
            xtrain, ytrain = x[train_inds], y[train_inds]
            xtest, ytest = x[test_inds], y[test_inds]

            X_train = np.column_stack([xtrain, xtrain**2])
            X_test = np.column_stack([xtest,xtest**2])

            # Standardize features (zero mean, unit variance for each feature)
            X_mean = X_train.mean(axis=0)
            X_std = X_train.std(axis=0)

            X_std[X_std == 0] = 1  # safeguard to avoid division by zero for constant features
            X_train_norm = (X_train - X_mean) / X_std
            X_test_norm = (X_test - X_mean) / X_std

            # sentrer y
            ytrain_mean = ytrain.mean()
            ytrain_centered = ytrain - ytrain_mean

            n_train = len(ytrain)
            n_features = X_train_norm.shape[1]
            I = np.eye(n_features)

            theta_cf_Ridge = np.linalg.solve(
                X_train_norm.T @ X_train_norm + n_train * lmb * I, 
                X_train_norm.T @ ytrain_centered
                )

            # Må legge tilbake snittet slik at prediskjonene er på originalakse
            y_pred = X_test_norm @ theta_cf_Ridge + ytrain_mean
            # Så kan vi sammenligne nå som prediksjonen er på samme akse på ytest
            mse = np.mean((y_pred - ytest)**2)
            score_KFold[i,j] = mse 
    mse_KFold = score_KFold.mean(axis=1)
    best_index = np.argmin(mse_KFold)
    best_lambda = lambdas[best_index]
    return mse_KFold, best_index, best_lambda

mse_KFold, best_index, best_lambda = cross_val(X_norm, y_centered)

print("Beste lambda:", best_lambda)
print("Laveste kryssvaliderte MSE:", mse_KFold[best_index])

Beste lambda: 0.001
Laveste kryssvaliderte MSE: 0.3614819671736032


This computes the Ridge and OLS regression coefficients directly. The identity
matrix $\boldsymbol{I}$ has the same size as $\boldsymbol{X}^T\boldsymbol{X}$
and adds $n\lambda$ to its diagonal for Ridge regression. We then solve the
linear system (prefer `np.linalg.solve` or `np.linalg.pinv` to an explicit
inverse). The result for $\boldsymbol{\theta}$ is a NumPy array of shape
`(n_features,)` containing the fitted parameters.

### 3a)

Finalise, in the above code, the OLS and Ridge regression determination of
the optimal parameters $\boldsymbol{\theta}$. Compare with
`Ridge(alpha=n*lam, fit_intercept=False)` from `scikit-learn`.

### 3b)

Explore the results as functions of different values of the hyperparameter
$\lambda$. See for example exercise 4 from week 36, and Section 3.14 of the
lecture notes for how $\lambda$ should be chosen in practice.

## Exercise 4: implementing the simplest form of gradient descent

Alternatively we can fit the model by gradient descent, Eq. (4.15) of the
lecture notes,

$$
\boldsymbol{\theta}_{k+1} = \boldsymbol{\theta}_k - \eta\,\nabla_{\boldsymbol{\theta}} C(\boldsymbol{\theta}_k).
$$

This is useful to visualise the iterative convergence, and it is necessary
when $n$ and $p$ are so large that the closed form is too slow or too
memory-hungry — or when, as for the Lasso, logistic regression and neural
networks, no closed form exists. Use the gradients of exercise 2 and set up,
using the template below, your own gradient descent code for OLS and Ridge
regression.

### 4a)

Write first a gradient descent code for OLS only, using the above template.
Discuss the results as functions of the learning rate and the number of
iterations: for $\eta \in \{0.01, 0.1, 0.5, 0.8\}$, how many iterations are
needed to reach the closed-form solution of exercise 3 to a given accuracy,
say $\|\boldsymbol{\theta}_k - \hat{\boldsymbol{\theta}}\|_2 < 10^{-6}$?
Plot this distance against the iteration number on a logarithmic scale.

### 4b)

Write then a similar code for Ridge regression. Add a stopping criterion
based on the number of iterations *and* on the size of the gradient (or on
the difference between the new and old $\boldsymbol{\theta}$). How would you
define a stopping criterion, and what goes wrong with a fixed number of
iterations alone?

In [217]:
# Gradient descent parameters, learning rate eta first
#eta = 0.1
# Then the maximum number of iterations, and a tolerance for the stopping criterion
num_iters = 1000
tol = 1e-8

def gd_Ridge(X, y, eta, n, lam):
    n = len(y)
    # Initialize weights for gradient descent
    theta_gdRidge = np.zeros(n_features)
    for t in range(num_iters):
        # Compute gradients  Ridge
        grad_Ridge = (((2.0 / n) * X.T @ (X @ theta_gdRidge - y)) 
        + 2.0 * (lam * theta_gdRidge))         # 4.17
        # Update parameters theta
        theta_gdRidge = theta_gdRidge - eta * grad_Ridge
        # Stopping criterion: ?
        if np.linalg.norm(grad_Ridge) < 1.0e-8:
            break 

    return theta_gdRidge, t

def gd_OLS(X, y, eta, n, lam):
    n = len(y)
    # Initialize weights for gradient descent
    theta_gdOLS = np.zeros(n_features)
    for t in range(num_iters):
        # Compute gradients for OLS
        grad_OLS = (2.0 / n) * X.T @ (X @ theta_gdOLS - y) # 4.13
        # Update parameters theta
        theta_gdOLS = theta_gdOLS - eta * grad_OLS
        # Stopping criterion: ?
        if np.linalg.norm(grad_OLS) < 1.0e-8:
            break 

    return theta_gdOLS, t


eta_list = [0.01, 0.1, 0.5, 0.8]
t_ols = []
t_ridge = []
lam = 0.5
theta_gd_OLS = np.zeros((len(eta_list),2))
theta_gd_Ridge = np.zeros((len(eta_list),2))
for i, eta in enumerate(eta_list):
    theta_gdOLS, to = gd_OLS(X_norm, y_centered, eta, n_features, lam)
    theta_gdRidge, tr = gd_Ridge(X_norm, y_centered, eta, n_features, lam)
    t_ols.append(to)
    t_ridge.append(tr)

Dev_Ridge = np.linalg.norm(theta_gdRidge - theta_closed_formRidge)
Dev_OLS = np.linalg.norm(theta_gdOLS - theta_closed_formOLS)

print(t_ols)
print(Dev_OLS)
print("-----")
print(t_ridge)
print(Dev_Ridge)
ECHO=False
print("""Closed-form Ridge coefficients: [-0.5608836   3.86236727]
Closed-form OLS coefficients: [-1.0773505  5.854734 ]
Skicit Ridge coefficients: [-0.5608836   3.86236727]""")


[999, 107, 10, 87]
3.146071979041389e-09
-----
[735, 64, 43, 999]
inf
Closed-form Ridge coefficients: [-0.5608836   3.86236727]
Closed-form OLS coefficients: [-1.0773505  5.854734 ]
Skicit Ridge coefficients: [-0.5608836   3.86236727]


/Users/timholmen/MachineLearningUiO/.venv/lib/python3.13/site-packages/numpy/linalg/_linalg.py:2767: RuntimeWarning: overflow encountered in dot
  sqnorm = x.dot(x)


For eta = 0.8, which is larger than lambda, the ridge solution diverges, the step size is to big, and the solution is never hit by the method. The limit is 
$$\eta < \frac{2}{\lambda_{max}(H)}$$

### 4c) The learning rate and the eigenvalues of the Hessian

Compute the eigenvalues $\lambda_{\max}$ and $\lambda_{\min}$ of the Hessian
matrix of exercise 2 (use `np.linalg.eigvalsh`). Section 4.5 of the lecture
notes shows that gradient descent converges if and only if
$\eta < 2/\lambda_{\max}$, Eq. (4.20), that the fastest convergence is
obtained for $\eta^* = 2/(\lambda_{\max} + \lambda_{\min})$, and that the
number of iterations grows with the condition number
$\kappa = \lambda_{\max}/\lambda_{\min}$, Eq. (4.21). Verify the bound
numerically: run your code at $0.99 \cdot 2/\lambda_{\max}$ and at
$1.02 \cdot 2/\lambda_{\max}$.

Repeat for a design matrix with the columns $x, x^2, \dots, x^5$
(standardised). What happens to $\kappa$, to $\eta_{\max}$ and to the number
of iterations, and why? Which of the two data sets does Ridge regression help
more, and in what sense?

In [218]:
# 4c) Learning rate and the eigenvalues of the Hessian
def Hessian_OLS(X, n):
    H_OLS = (2 / n) * X.T @ X
    return H_OLS

def Hessian_R(X, n, lamb):
    I = np.eye(X.shape[1])
    H_R = (2 / n) * X.T @ X + 2*lamb*I
    return H_R


H_OLS = Hessian_OLS(X_norm,n)
H_R = Hessian_R(X_norm, n, 0.5)
Eig_O = np.linalg.eigvalsh(H_OLS)
Eig_R = np.linalg.eigvalsh(H_R)
print(Eig_O)
print(Eig_R)
best_eta = 2/(Eig_R[1]+Eig_R[0])
bad_eta = 2/Eig_R[1]
print(eta)
print(bad_eta)

theta_gdOLS_good, to = gd_OLS(X_norm, y_centered, best_eta, n_features, lam)
theta_gdRidge_good, tr = gd_Ridge(X_norm, y_centered, best_eta, n_features, lam)

theta_gdOLS_bad, tob = gd_OLS(X_norm, y_centered, bad_eta, n_features, lam)
theta_gdRidge_bad, trb = gd_Ridge(X_norm, y_centered, 1.1*bad_eta, n_features, lam)

print("Good eta = 2/(lambda_max + lambda_min)")
print("Iterations good eta:", tr)
print("Deviance good eta", np.linalg.norm(theta_gdRidge_good - theta_closed_formRidge))
print("bad eta >= 2/lambda_max")
print("Iterations bad eta:", trb)
print("Deviance 1.1*bad_eta", np.linalg.norm(theta_gdRidge_bad - theta_closed_formRidge))
print("Divergence!")


[1.76307062 2.23692938]
[2.76307062 3.23692938]
0.8
0.61786952002118
Good eta = 2/(lambda_max + lambda_min)
Iterations good eta: 9
Deviance good eta 3.684351152746472e-11
bad eta >= 2/lambda_max
Iterations bad eta: 999
Deviance 1.1*bad_eta 3.5435633685192004e+79
Divergence!



### 4d) Checking the gradient with automatic differentiation

Project 1, part e), asks you to compute the gradient in two independent ways:
analytically, and by automatic differentiation. Write the OLS and Ridge cost
functions as ordinary Python functions of $\boldsymbol{\theta}$ using
`jax.numpy`, obtain their gradients with `jax.grad`, and compare with your
analytical gradients from exercise 2 at a random $\boldsymbol{\theta}$
(Section 4.14.7 of the lecture notes; remember
`jax.config.update("jax_enable_x64", True)`). Run your gradient descent code
with the automatic gradient and check that you obtain the same iterates.

In one or two sentences: why is automatic differentiation neither symbolic
nor numerical (finite-difference) differentiation, and what does a gradient
cost relative to one evaluation of the cost function (Section 4.14.4)?

In [ ]:

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
from jax import grad

def cost_OLS(theta, X, y):
    return jnp.sum((X @ theta - y)**2 / len(y))

def cost_Ridge(theta, X, y, lam):
    return (
        jnp.sum((X @ theta - y)**2) / len(y)
        + lam * jnp.sum(theta**2)
    )


def analytical_grad_OLS(theta,X,y):
    return (2.0 / len(y)) * X.T @ (X @ theta - y)
def analytical_grad_Ridge(theta, X, y, lam):
    return (
        (2.0 / len(y)) * X.T @ (X @ theta - y)
        + 2.0 * lam * theta)

theta_test = rng.standard_normal(n_features)

grad_OLS_ad = np.asarray(
    grad(cost_OLS)(jnp.asarray(theta_test),
                   jnp.asarray(X_norm),
                   jnp.asarray(y_centered))
)
grad_OLS_analytical = analytical_grad_OLS(
    theta_test, X_norm, y_centered
)

grad_Ridge_ad = np.asarray(
    grad(cost_Ridge)(jnp.asarray(theta_test),
                     jnp.asarray(X_norm),
                     jnp.asarray(y_centered),
                     lam)
)
grad_Ridge_analytical = analytical_grad_Ridge(
    theta_test, X_norm, y_centered, lam
)

print("OLS error:",
      np.max(np.abs(grad_OLS_ad - grad_OLS_analytical)))

print("Ridge error:",
      np.max(np.abs(grad_Ridge_ad - grad_Ridge_analytical)))
    




OLS error: 3.552713678800501e-15
Ridge error: 3.552713678800501e-15


In [ ]:
#import sys
#!{sys.executable} -m pip install -U "jax[cpu]"

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/pty.py:95: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


  Using cached jax-0.11.1-py3-none-any.whl.metadata (13 kB)
INFO: pip is looking at multiple versions of jax[cpu] to determine which version is compatible with other requirements. This could take a while.
  Using cached jax-0.11.0-py3-none-any.whl.metadata (13 kB)
  Using cached jax-0.10.2-py3-none-any.whl.metadata (13 kB)
  Using cached jax-0.10.1-py3-none-any.whl.metadata (13 kB)
  Using cached jax-0.10.0-py3-none-any.whl.metadata (13 kB)
  Using cached jax-0.9.2-py3-none-any.whl.metadata (13 kB)
  Using cached jax-0.9.1-py3-none-any.whl.metadata (13 kB)
  Using cached jax-0.9.0.1-py3-none-any.whl.metadata (13 kB)
INFO: pip is still looking at multiple versions of jax[cpu] to determine which version is compatible with other requirements. This could take a while.
  Using cached jax-0.9.0-py3-none-any.whl.metadata (13 kB)
  Using cached jax-0.8.3-py3-none-any.whl.metadata (13 kB)
  Using cached jax-0.8.2-py3-none-any.whl.metadata (13 kB)
  Using cached jax-0.8.1-py3-none-any.whl.metada

## Exercise 5: Ridge regression and a new synthetic data set

We create a synthetic linear regression data set with a sparse underlying
relationship: many features, of which only a few contribute to the target. We
use 10 features with only 3 non-zero weights in the true model, so that the
target is a linear combination of a few features (with known coefficients)
plus random noise. The steps are:

* decide on the number of samples and features (100 samples, 10 features);
* define the *true* coefficient vector with mostly zeros, for example
  $\boldsymbol{\theta}_{\mathrm{true}} = [5.0, -3.0, 0, 0, 0, 0, 2.0, 0, 0, 0]$,
  meaning only features 0, 1 and 6 have a real effect on $y$;
* sample the feature matrix $\boldsymbol{X}$ from a standard normal
  distribution, so that the features are roughly centred around 0;
* compute $\boldsymbol{y} = \boldsymbol{X}\boldsymbol{\theta}_{\mathrm{true}} + \boldsymbol{\varepsilon}$
  with Gaussian noise $\boldsymbol{\varepsilon}$.

In [230]:
import numpy as np

# Set random seed for reproducibility
np.random.seed(0)

# Define dataset size
n_samples = 100
n_features = 10

# Define true coefficients (sparse linear relationship)
theta_true = np.array([5.0, -3.0, 0.0, 0.0, 0.0, 0.0, 2.0, 0.0, 0.0, 0.0])

# Generate feature matrix X (n_samples x n_features) with random values
X = np.random.randn(n_samples, n_features)  # standard normal distribution

# Generate target values y with a linear combination of X and theta_true, plus noise
noise = 0.5 * np.random.randn(n_samples)    # Gaussian noise
y = X @ theta_true + noise

# Standardize features
X_mean = X.mean(axis=0)
X_std = X.std(axis=0)
X_std[X_std == 0] = 1
X_norm = (X - X_mean) / X_std 
# Center the target to zero mean
y_mean = y.mean()
y_centred = y - y_mean

lmb = 0.1               # Which lambda?
I = np.eye(n_features)
                        # Analytical Solution
def theta_closed_OLS(X, y):
    return np.linalg.solve(X.T @ X, X.T @ y)
def theta_closed_Ridge(X, y, lmb):
    return np.linalg.solve(X.T @ X + n * lmb * I, X.T @ y)

mse_KFold, best_index, best_lmb = cross_val(X_norm, y_centred)

theta_R = theta_closed_Ridge(X_norm, y_centred, best_lmb)
theta_O = theta_closed_OLS(X_norm, y_centred)

#print(theta_true)
#print(np.round(theta_R,2))
#print(np.round(theta_O,2))
#                       # Gradient Descent
def gd_Ridge_1(X, y, lam):
    n = len(y)
    # Calculate best Eta from Eig.Vals of Hessian
    Eig = np.linalg.eigvalsh(Hessian_R(X, n, lam))
    kappa = np.max(Eig)/np.min(Eig)
    eta = 2/(np.min(Eig) + np.max(Eig))
    n_features = np.shape(X[1])
    # Initialize weights for gradient descent
    theta_gdRidge = np.zeros(n_features)
    for t in range(num_iters):
        # Compute gradients  Ridge
        grad_Ridge = (((2.0 / n) * X.T @ (X @ theta_gdRidge - y)) 
        + 2.0 * (lam * theta_gdRidge))         # 4.17
        # Update parameters theta
        theta_gdRidge = theta_gdRidge - eta * grad_Ridge
        # Stopping criterion: ?
        if np.linalg.norm(grad_Ridge) < 1.0e-8:
            break 

    return theta_gdRidge, t, kappa, Eig

def gd_OLS_1(X, y):
    n = len(y)
    # Calculate best Eta from Eigenvalues of Hessian matrice
    Eig = np.linalg.eigvalsh(Hessian_OLS(X, n))
    kappa = np.max(Eig)/np.min(Eig)
    eta = 2/(np.min(Eig) + np.max(Eig))

    # Initialize weights for gradient descent
    theta_gdOLS = np.zeros(n_features)
    for t in range(num_iters):
        # Compute gradients for OLS
        grad_OLS = (2.0 / n) * X.T @ (X @ theta_gdOLS - y) # 4.13
        # Update parameters theta
        theta_gdOLS = theta_gdOLS - eta * grad_OLS
        # Stopping criterion: ?
        if np.linalg.norm(grad_OLS) < 1.0e-8:
            break 

    return theta_gdOLS, t, kappa, Eig

theta_gdOLS, tO, Cn_O, eig1 = gd_OLS_1(X, y)
theta_gdR, tR, Cn_R, eig2 = gd_Ridge_1(X, y, best_lmb)
print("Ridge Analytic:")
print(np.round(theta_R,2))
print("OLS Analytic:")
print(np.round(theta_O,2))
print("-----")
print(theta_true)
print("-----")
print("OLS Gradient Descent")
print(np.round(theta_gdOLS,2), "iterations",tR)
print("Ridge Gradient Descent")
print(np.round(theta_gdR,2), "iterations:",tR)
print("-------")
print(f"Condition number OLS: {Cn_O:.2}")
print(f"Condition number Ridge: {Cn_R:.2}")



Ridge Analytic:
[ 5.01 -2.88 -0.02  0.15 -0.07 -0.05  1.76  0.01  0.04 -0.05]
OLS Analytic:
[ 5.03 -2.89 -0.02  0.15 -0.07 -0.04  1.77  0.    0.05 -0.05]
-----
[ 5. -3.  0.  0.  0.  0.  2.  0.  0.  0.]
-----
OLS Gradient Descent
[ 5.01 -3.   -0.02  0.14 -0.07 -0.04  2.06  0.    0.04 -0.05] iterations 24
Ridge Gradient Descent
[ 4.99 -2.99 -0.02  0.15 -0.08 -0.05  2.05  0.    0.04 -0.05] iterations: 24
-------
Condition number OLS: 2.4
Condition number Ridge: 2.4


This code produces a data set where only features 0, 1 and 6 significantly
influence $\boldsymbol{y}$; the rest have zero true coefficient. Feature 0 has
a true weight of $5.0$, feature 1 of $-3.0$ and feature 6 of $2.0$, so the
expected relationship is

$$
y \approx 5x_0 - 3x_1 + 2x_6 + \text{noise}.
$$

You can remove the noise if you wish.

Fit the above data set using OLS and Ridge regression with the analytical
expressions and with your own gradient descent code (scale the data as in
exercise 1). If everything works, the learned coefficients should be close to
the true values $[5.0, -3.0, 0, \dots, 2.0, \dots]$. Due to regularisation
and noise the learned values will not exactly equal the true ones, but they
should be in the same ballpark. Which method, OLS or Ridge, gives the best
results — and best in which sense? Compute the condition number of the
Hessian for this data set: why does gradient descent converge so much more
easily here than for the degree-5 polynomial of exercise 4c)?

## Exercise 6: adding momentum

Part f) of Project 1 asks you to extend your gradient descent code with
momentum and, next week, with the adaptive methods AdaGrad, RMSProp and Adam.
Start here with momentum, Eq. (4.23) of the lecture notes,

$$
\boldsymbol{v}_{k+1} = \gamma\boldsymbol{v}_k + \eta\nabla_{\boldsymbol{\theta}} C(\boldsymbol{\theta}_k),
\qquad
\boldsymbol{\theta}_{k+1} = \boldsymbol{\theta}_k - \boldsymbol{v}_{k+1},
$$

which is one line more than the code of exercise 4. Use the degree-5
polynomial data of exercise 4c), where plain gradient descent is slow, and
the learning rate $\eta = 0.9 \cdot 2/\lambda_{\max}$.

1. Run $\gamma \in \{0, 0.5, 0.9\}$ and count the iterations needed to reach
   $\|\boldsymbol{\theta}_k - \hat{\boldsymbol{\theta}}\|_2 < 10^{-6}$. Plot
   the three convergence curves on a logarithmic scale.
2. Section 4.6 shows that, with optimally tuned parameters, momentum reduces
   the iteration count from $\propto\kappa$ to $\propto\sqrt\kappa$,
   Eq. (4.26). How large a reduction do you observe with the fixed $\eta$
   above, and how does it compare with $\sqrt\kappa$? Try the optimal
   momentum $\gamma^* = \big((\sqrt\kappa-1)/(\sqrt\kappa+1)\big)^2$.
3. Explain, in terms of Eqs. (4.24) and (4.25), why momentum speeds up the
   flat directions and damps the oscillations in the steep ones.
4. (Optional) Repeat with $\eta = 1.5 \cdot 2/\lambda_{\max}$, where plain
   gradient descent diverges. What happens with momentum, and why?

In [224]:
def runge_data(n=100, degree=6, noise=0.1, seed=2026):
    """Runge function 1/(1+25x^2) on [-1,1], standardised polynomial features, centred y."""
    rng = np.random.default_rng(seed)
    x = rng.uniform(-1.0, 1.0, n)
    y = 1.0 / (1.0 + 25.0 * x**2) + noise * rng.standard_normal(n)
    X = np.column_stack([x**k for k in range(1, degree + 1)])
    X_norm = (X - X.mean(axis=0)) / X.std(axis=0)
    return X_norm, y - y.mean()



In [ ]:
# Your momentum code here
""" Dette klarte jeg ikke.
gamma = [0, 0.5, 0.9]



def momentum_descent(X, y, eta, gamma, lam=0.0, num_iters = 5000, tol=1e-8):
    theta, v = np.zeros(X.shape[1]), np.zeros(X.shape[1])
    history = [theta.copy()]
    for t in range(num_iters):
        g, t1, kappa, eig = gd_Ridge_1(X, y, lam)
        v = gamma * v + eta * g 
        theta = theta - v 
        history.append(theta.copy())
        if np.linalg.norm(g) < tol:
            break
    return np.array(history), t + 1



def fig_momentum(filename="week37_momentum.pdf"):
    X, y = runge_data()
    theta_cf = theta_closed_OLS(X,y)
    eig = np.linalg.eigvalsh(Hessian_OLS(X, len(y)))
    lmax, lmin = eig.max(), eig.min()
    eta = 0.9 * 2.0 / lmax
    fig, ax = plt.subplots(figsize=(6.4, 4.0))
    for gamma, col, lab in ((0.0, "blue", r"plain GD, $\gamma=0$"),
                            (0.5, "yellow", r"momentum, $\gamma=0.5$"),
                            (0.9, "red", r"momentum, $\gamma=0.9$")):
        hist, _ = momentum_descent(X, y, eta, gamma, num_iters=16000, tol=0.0)
        ax.semilogy(np.linalg.norm(hist - theta_cf, axis=1), color=col, lw=1.6, label=lab)
    ax.axhline(1e-6, color="grey", ls=":", lw=1)
    ax.set_xlabel("iteration $k$")
    ax.set_ylabel(r"$\|\boldsymbol{\theta}_k-\hat{\boldsymbol{\theta}}_{\mathrm{OLS}}\|_2$")
    ax.set_title(rf"Runge function, degree 6: $\kappa={lmax/lmin:.0f}$, "
                 rf"$\eta=0.9\cdot2/\lambda_{{\max}}$", fontsize=10)
    ax.legend(fontsize=9)
    fig.tight_layout()
    fig.savefig(filename, bbox_inches="tight")
    plt.close(fig)


#print(f"Runge, degree 6: lambda_max = {lmax:.3f}, lambda_min = {lmin:.2e}, kappa = {lmax/lmin:.0f}")

for gamma in (0.0, 0.5, 0.9):
    hist, _ = momentum_descent(X, y, eta, gamma, num_iters=20000, tol=0.0)
    dist = np.linalg.norm(hist - theta_cf, axis=1)
    first = np.argmax(dist < 1e-6) if np.any(dist < 1e-6) else None
    print(f"   gamma = {gamma}: first iteration with |theta - closed form| < 1e-6: {first}")
fig_momentum("week37_momentum.png")
Image("week37_momentum.png")"""

   gamma = 0.0: first iteration with |theta - closed form| < 1e-6: None
   gamma = 0.5: first iteration with |theta - closed form| < 1e-6: None
   gamma = 0.9: first iteration with |theta - closed form| < 1e-6: None


NameError: name 'Image' is not defined

## Insights you should have gained this week — a self-check

Before you hand in, go through the list below without looking at the notes.
Each row is a question we may ask in the Tuesday session or on Project 1; the
right-hand column is the insight it tests, in one sentence.

| Ask yourself | The insight it tests |
|---|---|
| My loop stopped after its allowed iterations with a small but nonzero gradient. Has it converged? | No. Along the slowest direction the error contracts by $\lvert 1-\eta\lambda_{\min}\rvert$ per step, Eq. (4.19), and can still be $\|\nabla C\|/\lambda_{\min}$ large. Stop on the gradient (or on $\Delta\boldsymbol{\theta}$), never on a count alone. |
| Why does gradient descent care about the scaling of the columns when the closed form does not? | The closed form solves the normal equations in one go; gradient descent takes the same step in every direction, and the column scales set the eigenvalue spread $\kappa(\boldsymbol{H}) = \kappa_2(\boldsymbol{X})^2$, Eq. (4.22), hence the iteration count. Same minimum, different road. |
| Ridge converged in far fewer iterations than OLS. Did it find a better OLS solution? | No — it found the exact solution of a *different* problem, Eq. (4.16). The penalty lifts $\lambda_{\min}$, Eq. (4.17), which shortens the road, but the destination moved: bias traded for variance. |
| The same code diverged at $\eta = 0.5$ on one data set and converged at $\eta = 0.8$ on another. Why? | Stability requires $\eta < 2/\lambda_{\max}$, Eq. (4.20), and $\lambda_{\max}$ is a property of $\boldsymbol{X}$, not of the algorithm. Compute it — `np.linalg.eigvalsh(2/n * X.T @ X).max()` — before you choose $\eta$. |
| The closed form and my converged gradient descent disagree in the second decimal for Ridge. Which is wrong? | Possibly neither: with the cost $\frac{1}{n}\|\boldsymbol{X}\boldsymbol{\theta}-\boldsymbol{y}\|^2 + \lambda\|\boldsymbol{\theta}\|^2$ the closed form has $n\lambda\boldsymbol{I}$, Eq. (3.95), and `scikit-learn`'s `alpha` is $n\lambda$. Compare cost functions before comparing methods. |
| My JAX gradient agrees with my analytical gradient to $10^{-15}$. What have I proved? | That your derivation matches the cost you *wrote down* — an algebra error would show as $10^{-2}$. Not that the cost is the one you meant (a missing $1/n$, a penalised intercept): both gradients would then agree and both would be wrong. |
| Momentum converged where plain gradient descent diverged. Is the learning rate now unimportant? | No. The stability edge moves to $2(1+\gamma)/\lambda_{\max}$, and the best $\gamma$ is tied to $\kappa$ through Eq. (4.26); too much momentum rings for ever. Two data-dependent knobs instead of one. |
| Why does momentum help, in one sentence? | It adds up gradients that keep pointing the same way (the flat directions, effective step $\eta/(1-\gamma)$, Eq. (4.25)) and cancels gradients that alternate in sign (the steep direction), so the iteration count scales with $\sqrt\kappa$ instead of $\kappa$. |
| I swap the OLS cost for a neural-network cost and keep `jax.grad`. What changes? | In the code: only the definition of the cost. In the theory: convexity is gone, so a stationary point need not be the global minimum (Section 4.1) and the bound of Eq. (4.20) is only local — good behaviour becomes empirical, not a theorem. |
| Why reverse mode for training, and what is its price? | One scalar output and $p$ inputs: reverse mode returns the full gradient for two or three cost evaluations, whatever $p$ (the cheap gradient principle, Section 4.14.4); forward mode or finite differences would need $p$ sweeps. The price is the tape — every intermediate value is stored until the backward sweep has used it: memory, not time. |
| Why not just use finite differences to check the gradient, and be done with it? | For a *quadratic* cost the central difference is exact up to rounding, so it would pass; for anything else its best accuracy is about $10^{-11}$ at $h\approx10^{-6}$, Eq. (4.46), and it costs $2p$ evaluations. Use it as a check at $h \approx 10^{-5}$, not as the gradient. |

### Practicalities

Deliver your solutions (a single Jupyter notebook, with the analytical parts
either typeset in Markdown/LaTeX or as a scanned handwritten note) in Canvas
before **Friday September 11 at midnight**. Collaboration is encouraged — list
your collaborators.